# 01 · Limpeza dos dados

Projeto: prever o preço pedido de apartamentos em Lisboa com Machine Learning.

| Notebook | O que faz |
|---|---|
| 01 · Limpeza | Escolhe as colunas úteis e corrige erros |
| 02 · Exploração | Gráficos para conhecer os dados |
| 03 · Feature engineering | Cria e prepara as variáveis |
| 04 · Modelos | Treina e compara três modelos |
| 05 · Avaliação | Mede o erro em casas que o modelo nunca viu |
| 06 · Previsão | Estima o preço de uma casa |

Os dados não são publicados: os notebooks mostram só resumos e gráficos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DADOS = Path("..") / "data"   # a pasta data fica ao lado da pasta notebooks

## 1. Ler os dados

Usamos o ficheiro `dataset_*.csv` mais recente da pasta `data/raw`.

In [2]:
ficheiro = sorted((DADOS / "raw").glob("dataset_*.csv"))[-1]   # o último por ordem alfabética = o mais recente
raw = pd.read_csv(ficheiro, low_memory=False)
print(f"{raw.shape[0]} anúncios e {raw.shape[1]} colunas")

1661 anúncios e 453 colunas


## 2. Ficar só com as colunas úteis

A exportação tem mais de 450 colunas (fotos, contactos, etc.). Ficamos com as que descrevem a casa e damos-lhes nomes em português. Nada de pessoal (nomes, telefones, moradas) entra no projeto.

In [3]:
colunas = {
    "propertyCode": "id", "price": "preco", "size": "area_m2", "rooms": "quartos",
    "bathrooms": "casas_banho", "floor": "andar", "status": "estado", "propertyType": "tipo",
    "municipality": "freguesia", "latitude": "latitude", "longitude": "longitude",
    "operation": "operacao", "hasLift": "elevador", "newDevelopment": "empreendimento_novo",
    "features/hasTerrace": "terraco", "features/hasSwimmingPool": "piscina",
    "features/hasGarden": "jardim", "features/hasAirConditioning": "ar_condicionado",
    "features/hasBoxRoom": "arrecadacao", "parkingSpace/hasParkingSpace": "estacionamento",
}
df = raw[list(colunas)].rename(columns=colunas)
df.isna().mean().mul(100).round(1).sort_values(ascending=False).head(8)   # % de valores em falta

estacionamento     54.6
piscina            45.6
jardim             43.0
terraco            40.6
arrecadacao        37.3
andar              16.9
ar_condicionado     9.8
elevador            6.0
dtype: float64

Os extras (piscina, terraço, estacionamento...) faltam em 37% a 55% dos anúncios. Tratamos disso no passo 5.

## 3. Só apartamentos à venda

In [4]:
df = df[df["tipo"].isin(["flat", "duplex", "penthouse", "studio"]) & (df["operacao"] == "sale")]
df = df.drop(columns="operacao")
print(f"{len(df)} anúncios")

1661 anúncios


## 4. Valores impossíveis e duplicados

- Áreas fora de 20 a 600 m² e mais de 8 quartos são erros: saem.
- 0 casas de banho é suspeito: fica como valor em falta.
- A mesma casa anunciada duas vezes (mesma área, quartos, casas de banho, preço e localização) só conta uma vez.

In [5]:
df = df[df["area_m2"].between(20, 600) & (df["quartos"] <= 8)]
df["casas_banho"] = df["casas_banho"].replace(0, np.nan)

antes = len(df)
df = df.drop_duplicates(subset="id")
# Mesma casa: mesmas características e coordenadas iguais até 3 casas decimais (cerca de 100 m)
chave = df[["area_m2", "quartos", "casas_banho", "preco"]].assign(lat=df["latitude"].round(3),
                                                                  lon=df["longitude"].round(3))
df = df[~chave.duplicated()]   # duplicated() = True a partir da segunda vez que aparece
df = df.drop(columns="id")
print(f"Removidos {antes - len(df)} duplicados. Ficam {len(df)} anúncios.")

Removidos 29 duplicados. Ficam 1623 anúncios.


## 5. Corrigir formatos

- O andar vem em texto, com códigos: `bj` = rés-do-chão (0), `en` = entresol (0,5), `st`/`ss` = cave (-1).
- Os extras vêm como verdadeiro/falso e passam a 1/0. Quando o anúncio não diz nada, assumimos que a casa não tem (0): quem tem piscina ou garagem costuma dizê-lo.

In [6]:
df["andar"] = pd.to_numeric(df["andar"].replace({"bj": 0, "en": 0.5, "st": -1, "ss": -1}), errors="coerce")

extras = ["elevador", "empreendimento_novo", "terraco", "piscina", "jardim",
          "ar_condicionado", "arrecadacao", "estacionamento"]
for coluna in extras:
    df[coluna] = df[coluna].map({True: 1, False: 0}).fillna(0).astype(int)

falta = df.isna().sum()
falta[falta > 0]   # as colunas onde ainda falta alguma coisa

casas_banho      4
andar          275
estado          20
dtype: int64

Ainda falta o andar em cerca de 17% das casas e o estado em poucas. Não apagamos estas casas: o modelo preenche os vazios (notebook 04).

## 6. Gravar

In [7]:
(DADOS / "processed").mkdir(exist_ok=True)
df.to_csv(DADOS / "processed" / "dados_limpos.csv", index=False)
print(f"Gravado: {len(df)} anúncios e {df.shape[1]} colunas")

Gravado: 1623 anúncios e 18 colunas
